# De Levenscyclus van een Zaak

**Thema:** Dossiers en parlementaire zaken.

**Kernvragen:** Wat zijn de verschillende soorten zaken (`zaak.soort`), hoe lang duurt het voordat een zaak wordt afgedaan (`afgedaan = true`), en welke zaken zijn gemarkeerd als groot project?

Dit notebook verkent o.a. de silver-tabellen `zaak`, `zaak_activiteit`, `zaak_gerelateerd_vanuit` en `zaak_vervangen_vanuit`.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)


def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

WindowsPath('C:/Users/boate/tweedekamer.duckdb')

## Onderwerp 1

laat de feitentabel zien <br/>
hoeveel zaken worden er afgedaan en niet afgedaan? <br/>
Wat zijn de top 3 zaken die het meest worden afgewezen? <br/>

In [14]:
df = query_df("""
SELECT *
FROM silver.zaak""")
df

,id,nummer,soort,onderwerp,gestart_op,organisatie,titel,citeertitel,alias,grondslagvoorhang,termijn,vergaderjaar,volgnummer,status,afgedaan,groot_project,kabinetsappreciatie,kamerstukdossier_id,gewijzigd_op,api_gewijzigd_op
0,00067ff6-08fc-464a-afe0-42d05ba54b9a,2018Z08441,Brief regering,Nader antwoord op een vraag van het lid Midden...,2018-05-08,Tweede Kamer,Topinkomens,None,None,None,NaT,2017-2018,110,Vrijgegeven,True,False,None,17ca2762-feca-4fca-985f-cbde54713098,2024-02-19 11:59:02.367000+01:00,2024-02-19 16:30:36.388540+01:00
1,000e6eb4-dcde-4084-b091-66b503bc0371,2026Z05571,Motie,Motie van het lid Edgar Mulder over afzien van...,2026-03-19,Tweede Kamer,Vaststelling van de begrotingsstaten van het M...,None,None,None,NaT,2025-2026,56,Vrijgegeven,True,False,Ontraden,40a3f2dd-5bea-490e-84a1-892343c43695,2026-03-24 18:26:33.887000+01:00,2026-03-24 17:26:36.589162+01:00
2,000f5649-bbff-438a-be6d-6077705264d6,2011Z22403,Motie,Motie Venrooy-van Ark c.s. over toegang innova...,2011-11-09,Tweede Kamer,Toekomst AWBZ,None,None,None,NaT,2011-2012,229,Vrijgegeven,True,False,Niet beschikbaar bij moties en/of amendementen...,42c8fdab-879b-46af-89b5-73e28557deed,2024-02-19 11:59:02.367000+01:00,2024-02-19 14:01:11.582893+01:00
3,000f9a9e-6022-43a6-880a-65f67037bc34,2016Z22359,Motie,Gewijzigde motie van het lid Knops over veroor...,2016-11-29,Tweede Kamer,Mensenrechten in het buitenlands beleid,None,None,None,NaT,2016-2017,167,Vrijgegeven,True,False,Niet beschikbaar bij gewijzigde moties en/of a...,d54fca90-1b1b-4021-a4d5-4ffd1ee2b181,2024-02-19 11:59:02.367000+01:00,2024-02-19 13:20:43.043761+01:00
4,001e9a86-9917-40f5-a0f1-be06df365429,2017Z09213,Brief regering,Uitstel toezending informatie over de stand va...,2017-06-28,Tweede Kamer,Deelnemingenbeleid Rijksoverheid,None,None,None,NaT,2016-2017,268,Vrijgegeven,True,False,None,d10dfbb2-1a99-49f6-9dbb-8d702108d0a9,2024-02-19 11:59:02.367000+01:00,2024-02-19 15:40:25.103099+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
303092,f3ef2820-8657-43f3-8cb7-29b2f8802aa9,2026Z20099,Brief regering,Jaarlijkse rapportage stand spreiding rijkswer...,2026-09-25,Tweede Kamer,None,None,None,None,NaT,2026-2027,-1,Vrijgegeven,False,False,None,None,2026-09-25 18:19:21.093000+02:00,2026-09-25 16:19:38.897730+02:00
303093,f3f27e9c-3e90-473f-a0f0-f7e49872b0ee,2026Z16199,Schriftelijke vragen,Het bericht ‘’Hulporganisaties stoppen met wer...,2026-07-14,Tweede Kamer,None,None,None,None,NaT,2025-2026,-1,Vrijgegeven,True,False,None,None,2026-07-20 18:21:24.440000+02:00,2026-09-21 10:27:32.148549+02:00
303094,f3f8fe6c-69d2-45b9-a3b9-2eda1f4c3b21,2025Z07706,Position paper,Position paper Cito t.b.v. rondetafelgesprek T...,2025-04-16,Tweede Kamer,None,None,None,None,NaT,2024-2025,-1,Vrijgegeven,True,False,None,None,2025-04-25 10:09:49.813000+02:00,2025-04-25 08:09:53.293046+02:00
303095,f409d518-8958-48fe-b591-bec1677fa54e,2009Z00258,Schriftelijke vragen,Het door meerderjarigen betrekken van minderja...,2009-01-12,Tweede Kamer,None,None,None,None,NaT,2008-2009,-1,Vrijgegeven,True,False,None,None,2024-02-19 11:59:02.367000+01:00,2024-02-19 15:01:31.918813+01:00


In [16]:
df = query_df("""
select count(id)
from silver.zaak
where afgedaan = True
""")
df

,count(id)
0,297407


In [41]:
df = query_df("""
select count(id)
from silver.zaak
where afgedaan = False
""")
df

,count(id)
0,5690


In [42]:
df = query_df(""" 
select soort, titel
from silver.zaak
where afgedaan = False""")
df

,soort,titel
0,Brief regering,Verbetering verantwoording en begroting
1,Brief regering,Decoratiestelsel
2,Brief regering,Algemene Vergadering der Verenigde Naties
3,Brief regering,Toekomstvisie agrarische sector
4,Initiatiefnota,Initiatiefnota van de leden Paternotte en Bosw...
...,...,...
5685,Verzoek bij commissie-regeling van werkzaamheden,None
5686,Brief van lid/fractie/commissie,None
5687,Schriftelijke vragen,None
5688,Brief commissie,None


In [49]:
df = query_df("""
select count(*) as "aantal zaken zonder titel"
from silver.zaak
where titel IS NULL""")
df

,aantal zaken zonder titel
0,113329


In [50]:
df = query_df("""
select count(*) as "hoeveelheid zaken"
from silver.zaak""")
df

,hoeveelheid zaken
0,303097


In [40]:
df = query_df("""
select count(soort), soort
from silver.zaak
where afgedaan = False
group by soort
order by count(soort) desc""")
df

,count(soort),soort
0,3339,Brief regering
1,366,Brief commissie
2,265,Motie
3,260,Schriftelijke vragen
4,251,Brief van lid/fractie/commissie
5,198,Verzoek bij commissie-regeling van werkzaamheden
6,114,Wetgeving
7,100,Overig
8,93,Amendement
9,90,Position paper


In [52]:
df = query_df("""
select count(soort), soort
from silver.zaak
group by soort
order by count(soort) desc""")
df

,count(soort),soort
0,98245,Brief regering
1,65504,Motie
2,47208,Schriftelijke vragen
3,23531,Brief commissie
4,15210,Amendement
5,8616,Rondvraagpunt procedurevergadering
6,8423,Brief van lid/fractie/commissie
7,7648,Position paper
8,5192,Verzoek bij regeling van werkzaamheden
9,3370,Verzoek bij commissie-regeling van werkzaamheden


In [51]:
113329/303097

0.3739034038608102

### Conclusie Onderwerp 1
De enige opmerking die er gemaakt kan worden met de data van hierboven is dat er niet bij alle zaken een titel zit, dit geldt voor ongeveer 37% van de zaken. <br/> Voor de rest is er ook antwoord gegeven op de vraag: "Wat zijn de top 3 zaken die het meest worden afgewezen?", namelijk: <br/> <br/>Brief regering <br/> Brief commissie </br> Motie <br/><br/> dit komt natuurlijk doordat er ook meer van die zaken zijn, want er zijn een stuk minder zaken met de soort "Netwerkverkenning", wat wel logisch is.

## Onderwerp 2

...